In [2]:
import numpy as np
import pandas as pd
from itertools import product

In [3]:
### Define before run script

turbine = None

In [4]:
print(pd.read_parquet("../data/SCADA/scada_farm_2024-04_2026-04.parquet").columns.tolist())

['TimeStamp', 'WPSStatus', 'ActivePower', 'ActivePowerExport', 'DataOk', 'ActiveEnergy', 'ActivePowerMean']


In [5]:
print(pd.read_parquet("../data/SCADA/scada_turbines_2024-04_2026-04.parquet").columns.tolist())

['TimeStamp', 'StationId', 'wtc_ActPower_min', 'wtc_ActPower_mean', 'wtc_ActRegSt_endvalue', 'wtc_ActPower_Quality_endvalue', 'wtc_PitcPosA_mean', 'wtc_PowerRef_endvalue', 'wtc_PrWindSp_mean', 'wtc_SeWindSp_mean', 'wtc_AcWindSp_mean', 'wtc_AcWindSp_stddev', 'wtc_NacelPos_mean', 'wtc_BoostAct_mean', 'wtc_ActualWindDirection_mean', 'wtc_WTOperSt_endvalue', 'wtc_AmbieTmp_mean', 'wtc_ScInOper_timeon', 'wtc_ScReToOp_timeon', 'wtc_ScEnvSto_timeon', 'wtc_ScComSto_timeon', 'wtc_ScTurSto_timeon', 'wtc_ScGrdSto_timeon', 'wtc_IceRelay_timeon', 'wtc_PowerRed_timeon']


In [6]:
scada_tur = pd.read_parquet("../data/SCADA/scada_turbines_2024-04_2026-04.parquet")

In [7]:
scada_tur['StationId'].value_counts()

StationId
2304530    107564
2304525    107539
2304510    107536
2304518    107524
2304529    107523
2304519    107519
2304512    107510
2304527    107510
2304526    107508
2304528    107424
2304521    107408
2304515    107350
2304523    107184
2304522    107073
2304516    107021
2304524    107012
2304513    106929
2304520    106701
2304511    106413
2304514    106244
2304517    105366
Name: count, dtype: int64

In [8]:
#df = scada_tur[scada_tur['StationId'] == 2304517]
df = scada_tur.copy()

### Hill of Towie Wind Farm - SCADA Dataset Dictionary

This dataset contains 10-minute aggregated SCADA data from the commercial **Hill of Towie** wind farm (21 x Siemens SWT-2.3-VS-82 turbines in Scotland). The prefix `wtc_` stands for Wind Turbine Controller, and the `TimeStamp` indicates the end of each 10-minute UTC interval.

#### 1. Identification
* **`TimeStamp`**: Measurement timestamp in UTC format (end of the 10-minute interval).
* **`StationId`**: Unique identifier (ID) of an individual wind turbine in the park (e.g., `T01`).

#### 2. Power Production & Regulation
* **`wtc_ActPower_mean`**: Mean generated active power during the interval (primary target variable for power curves).
* **`wtc_ActPower_min`**: Minimum recorded active power within the interval.
* **`wtc_PowerRef_endvalue`**: Power reference (target) value at the end of the interval (grid curtailment/limits).
* **`wtc_ActRegSt_endvalue`**: Active power regulation status end value.
* **`wtc_ActPower_Quality_endvalue`**: Data quality/validity indicator for power measurements.
* **`wtc_PowerRed_endvalue` / `wtc_PowerRed_timeon`**: Power reduction metrics (time in seconds spent in reduced power mode within the interval).

#### 3. Aerodynamics, Wind & Positioning
* **`wtc_PrWindSp_mean`**: Mean wind speed measured by the primary nacelle anemometer.
* **`wtc_SeWindSp_mean`**: Mean wind speed measured by the secondary (backup) anemometer.
* **`wtc_AcWindSp_mean`**: Actual/accurate wind speed corrected for rotor induction effects.
* **`wtc_AcWindSp_stddev`**: Standard deviation of the actual wind speed (used for Turbulence Intensity - TI calculations).
* **`wtc_ActualWindDirection_mean`**: Mean actual wind direction (in degrees).
* **`wtc_NacelPos_mean`**: Mean nacelle position (azimuth) relative to true north.
* **`wtc_PitcPosA_mean`**: Mean rotor blade pitch angle (phase A / average).

#### 4. Environment & Operational Status
* **`wtc_AmbieTmp_mean`**: Mean ambient/outside air temperature (in °C).
* **`wtc_BoostAct_mean`**: Mean activity value of performance-optimizing "boost" modes.
* **`wtc_WTOperSt_endvalue`**: Wind turbine operational state at the end of the interval.

#### 5. State Duration Timers (in seconds / max 600s per interval)
* **`wtc_ScInOper_timeon`**: Time spent in normal, productive operation.
* **`wtc_ScReToOp_timeon`**: Time spent in start-up / transition to operation mode.
* **`wtc_ScEnvSto_timeon`**: Time stopped due to unfavorable environmental conditions (e.g., high wind cut-out).
* **`wtc_ScComSto_timeon`**: Time stopped by command (manual shutdown by operators/maintenance).
* **`wtc_ScTurSto_timeon`**: Time stopped due to internal turbine faults, alarms, or errors.
* **`wtc_ScGrdSto_timeon`**: Time stopped due to electrical grid issues or external grid curtailment.
* **`wtc_IceRelay_timeon`**: Time the ice relay protection was active (blade icing detection).

In [9]:
df.shape

(2249858, 25)

In [10]:
df.isnull().mean()

TimeStamp                        0.000000
StationId                        0.000000
wtc_ActPower_min                 0.000067
wtc_ActPower_mean                0.000067
wtc_ActRegSt_endvalue            0.000047
wtc_ActPower_Quality_endvalue    0.993020
wtc_PitcPosA_mean                0.000067
wtc_PowerRef_endvalue            0.000046
wtc_PrWindSp_mean                0.000067
wtc_SeWindSp_mean                0.000067
wtc_AcWindSp_mean                0.000067
wtc_AcWindSp_stddev              0.000067
wtc_NacelPos_mean                0.000045
wtc_BoostAct_mean                0.000047
wtc_ActualWindDirection_mean     0.000067
wtc_WTOperSt_endvalue            0.000046
wtc_AmbieTmp_mean                0.000067
wtc_ScInOper_timeon              0.000046
wtc_ScReToOp_timeon              0.000047
wtc_ScEnvSto_timeon              0.000047
wtc_ScComSto_timeon              0.000047
wtc_ScTurSto_timeon              0.000047
wtc_ScGrdSto_timeon              0.000047
wtc_IceRelay_timeon              0

In [11]:
df['wtc_BoostAct_mean'].value_counts()

wtc_BoostAct_mean
0.0    2249753
Name: count, dtype: int64

In [12]:
df['wtc_ActPower_Quality_endvalue'].value_counts()

wtc_ActPower_Quality_endvalue
3.0    15704
Name: count, dtype: int64

In [13]:
for col in df.columns:

    if len(df[col].value_counts()) <10:
        print("**********************")
        print("----------------------")
        
        print(df[col].value_counts())

**********************
----------------------
wtc_ActRegSt_endvalue
NotActive                                             2249753
Name: count, dtype: int64
**********************
----------------------
wtc_ActPower_Quality_endvalue
3.0    15704
Name: count, dtype: int64
**********************
----------------------
wtc_BoostAct_mean
0.0    2249753
Name: count, dtype: int64
**********************
----------------------
wtc_IceRelay_timeon
0.0    2249703
Name: count, dtype: int64


In [14]:
print(df.shape)
df["TimeStamp"] = pd.to_datetime(df["TimeStamp"])
df = df.drop_duplicates() #subset = ["TimeStamp"]).sort_values("TimeStamp")

print(df.shape)

(2249858, 25)
(2249810, 25)


In [15]:
if turbine is not None:
    df = df[df["StationId"] == turbine]
    assert df["StationId"].nunique() == 1 #"Set turbine == $SationId"

In [16]:
stop_columns = ["wtc_ScEnvSto_timeon", "wtc_ScComSto_timeon",
               "wtc_ScTurSto_timeon", "wtc_ScGrdSto_timeon"]


In [17]:
df['clear'] = (
    (df["wtc_ActPower_Quality_endvalue"] !=3)
    & (df[stop_columns]==0).all(axis=1)
    & (df["wtc_PowerRed_timeon"]==0)
    & (df["wtc_IceRelay_timeon"]==0)
    & (df[["wtc_ActPower_mean", "wtc_AcWindSp_mean"]].notna().all(axis=1))
)

In [18]:
print(f"There are 10-min clear intervals: {df['clear'].mean():.1%}")

There are 10-min clear intervals: 90.0%


In [19]:
df['hour'] = (df["TimeStamp"] - pd.Timedelta(minutes=10)).dt.ceil("h")

In [20]:
dfs = df[['TimeStamp' ,'StationId', 'wtc_AcWindSp_mean','wtc_ActualWindDirection_mean']][df['TimeStamp'].dt.minute==0]

In [21]:
df["E_10min_MWh"] = df["wtc_ActPower_mean"] / 1000 * (10 / 60)

In [22]:
df['wtc_ActPower_mean'].min(), df['wtc_ActPower_mean'].max()

(np.float32(-23.06583), np.float32(2304.833))

In [23]:
df['hour'][:20]

0    2024-04-01 00:00:00+00:00
1    2024-04-01 00:00:00+00:00
2    2024-04-01 00:00:00+00:00
3    2024-04-01 00:00:00+00:00
4    2024-04-01 00:00:00+00:00
5    2024-04-01 00:00:00+00:00
6    2024-04-01 00:00:00+00:00
7    2024-04-01 00:00:00+00:00
8    2024-04-01 00:00:00+00:00
9    2024-04-01 00:00:00+00:00
10   2024-04-01 00:00:00+00:00
11   2024-04-01 00:00:00+00:00
12   2024-04-01 00:00:00+00:00
13   2024-04-01 00:00:00+00:00
14   2024-04-01 00:00:00+00:00
15   2024-04-01 00:00:00+00:00
16   2024-04-01 00:00:00+00:00
17   2024-04-01 00:00:00+00:00
18   2024-04-01 00:00:00+00:00
19   2024-04-01 00:00:00+00:00
Name: hour, dtype: datetime64[us, UTC]

In [24]:
ws = df['wtc_AcWindSp_mean']
wd = np.deg2rad(df['wtc_ActualWindDirection_mean'])

df['u'] = -ws *np.sin(wd)
df['v'] = -ws *np.cos(wd)

sati = df.groupby(["hour", "StationId"]).agg(
    n_clear=("clear", "sum"),
    wind_mean=("wtc_AcWindSp_mean", "mean"),
    E_MWh=("E_10min_MWh", "sum"),
    u_mean=("u", "mean"),                
    v_mean=("v", "mean"),
    )

sati['wd_mean'] = np.rad2deg(np.arctan2(-sati["u_mean"], -sati["v_mean"])) % 360

In [25]:
dff = sati[sati['n_clear']==6]
dff.reset_index(inplace=True)

In [26]:
sati.shape

(375275, 6)

In [27]:
dfs.shape

(374983, 4)

In [28]:
dff.shape

(297862, 8)

In [29]:
dfs

,TimeStamp,StationId,wtc_AcWindSp_mean,wtc_ActualWindDirection_mean
0,2024-04-01 00:00:00+00:00,2304510,5.405419,124.667397
1,2024-04-01 00:00:00+00:00,2304511,4.477453,92.477280
2,2024-04-01 00:00:00+00:00,2304512,5.538813,106.480904
3,2024-04-01 00:00:00+00:00,2304513,5.255334,111.034401
4,2024-04-01 00:00:00+00:00,2304514,4.617242,84.922050
...,...,...,...,...
2249853,2026-05-01 00:00:00+00:00,2304526,6.505188,199.310806
2249854,2026-05-01 00:00:00+00:00,2304527,5.215073,184.301407
2249855,2026-05-01 00:00:00+00:00,2304528,4.173026,189.139603
2249856,2026-05-01 00:00:00+00:00,2304529,4.927575,176.301102


In [30]:

dff

,hour,StationId,n_clear,wind_mean,E_MWh,u_mean,v_mean,wd_mean
0,2024-04-01 01:00:00+00:00,2304510,6,4.182345,0.075824,-3.550668,2.193227,121.703377
1,2024-04-01 01:00:00+00:00,2304511,6,4.125554,0.061377,-4.112202,-0.235364,86.724220
2,2024-04-01 01:00:00+00:00,2304512,6,4.745492,0.124967,-4.642454,0.954168,101.614319
3,2024-04-01 01:00:00+00:00,2304513,6,5.337706,0.157387,-5.108343,1.523224,106.603714
4,2024-04-01 01:00:00+00:00,2304514,6,4.735353,0.101319,-4.650571,-0.833310,79.841293
...,...,...,...,...,...,...,...,...
297857,2026-04-30 23:00:00+00:00,2304524,6,1.754426,-0.009600,1.343055,0.034875,268.512543
297858,2026-04-30 23:00:00+00:00,2304525,6,2.490801,0.003791,1.122076,2.091549,208.212708
297859,2026-04-30 23:00:00+00:00,2304527,6,2.721987,-0.000137,1.341777,1.724683,217.882370
297860,2026-04-30 23:00:00+00:00,2304529,6,5.071109,0.284112,1.009518,4.913272,191.610855


In [31]:
dfs.rename(columns={"TimeStamp":"hour", "wtc_AcWindSp_mean":"wind_act",	"wtc_ActualWindDirection_mean":"wd_act"}, inplace=True)

In [32]:
final_df = pd.merge(
    dfs, dff, 
    on = ['hour', 'StationId'],
    how = "inner"
)

In [33]:
final_df.shape

(297862, 10)

In [34]:
final_df.head()

,hour,StationId,wind_act,wd_act,n_clear,wind_mean,E_MWh,u_mean,v_mean,wd_mean
0,2024-04-01 01:00:00+00:00,2304510,4.055212,120.949303,6,4.182345,0.075824,-3.550668,2.193227,121.703377
1,2024-04-01 01:00:00+00:00,2304511,5.062713,87.537354,6,4.125554,0.061377,-4.112202,-0.235364,86.724220
2,2024-04-01 01:00:00+00:00,2304512,5.411066,100.597198,6,4.745492,0.124967,-4.642454,0.954168,101.614319
3,2024-04-01 01:00:00+00:00,2304513,5.617938,105.831802,6,5.337706,0.157387,-5.108343,1.523224,106.603714
4,2024-04-01 01:00:00+00:00,2304514,4.995935,79.370956,6,4.735353,0.101319,-4.650571,-0.833310,79.841293


In [35]:
nwp_df = pd.read_csv("../data/nwp/open_meteo_single_runs.csv")
nwp_df['forecast_time'] = pd.to_datetime(nwp_df['forecast_time'], utc=True)

In [36]:
fdf = nwp_df.merge(
    final_df,
    left_on="forecast_time", right_on="hour",
    how="inner",                 
    )

In [37]:
for s, sid in product(np.arange(0,49), fdf.StationId.unique()):
    print("Za step: %s & Station_Id: %s, shape is %s" %(s, sid, fdf[(fdf['StationId']==2304511)&(fdf['fcst_step']==1)].shape))

Za step: 0 & Station_Id: 2304510, shape is (1275, 19)
Za step: 0 & Station_Id: 2304511, shape is (1275, 19)
Za step: 0 & Station_Id: 2304512, shape is (1275, 19)
Za step: 0 & Station_Id: 2304513, shape is (1275, 19)
Za step: 0 & Station_Id: 2304514, shape is (1275, 19)
Za step: 0 & Station_Id: 2304515, shape is (1275, 19)
Za step: 0 & Station_Id: 2304516, shape is (1275, 19)
Za step: 0 & Station_Id: 2304517, shape is (1275, 19)
Za step: 0 & Station_Id: 2304518, shape is (1275, 19)
Za step: 0 & Station_Id: 2304519, shape is (1275, 19)
Za step: 0 & Station_Id: 2304520, shape is (1275, 19)
Za step: 0 & Station_Id: 2304521, shape is (1275, 19)
Za step: 0 & Station_Id: 2304522, shape is (1275, 19)
Za step: 0 & Station_Id: 2304523, shape is (1275, 19)
Za step: 0 & Station_Id: 2304524, shape is (1275, 19)
Za step: 0 & Station_Id: 2304525, shape is (1275, 19)
Za step: 0 & Station_Id: 2304526, shape is (1275, 19)
Za step: 0 & Station_Id: 2304527, shape is (1275, 19)
Za step: 0 & Station_Id: 230

In [38]:
(1275/(2*(366+365)))*100

87.20930232558139

In [39]:
fdf.head()

,model,run_init,forecast_time,temperature_2m,wind_speed_10m,wind_speed_80m,wind_direction_10m,wind_direction_80m,fcst_step,hour,StationId,wind_act,wd_act,n_clear,wind_mean,E_MWh,u_mean,v_mean,wd_mean
0,ecmwf_ifs,2024-04-01 00:00:00,2024-04-01 01:00:00+00:00,1.7,2.06,4.98,129.0,110.0,1,2024-04-01 01:00:00+00:00,2304510,4.055212,120.949303,6,4.182345,0.075824,-3.550668,2.193227,121.703377
1,ecmwf_ifs,2024-04-01 00:00:00,2024-04-01 01:00:00+00:00,1.7,2.06,4.98,129.0,110.0,1,2024-04-01 01:00:00+00:00,2304511,5.062713,87.537354,6,4.125554,0.061377,-4.112202,-0.235364,86.724220
2,ecmwf_ifs,2024-04-01 00:00:00,2024-04-01 01:00:00+00:00,1.7,2.06,4.98,129.0,110.0,1,2024-04-01 01:00:00+00:00,2304512,5.411066,100.597198,6,4.745492,0.124967,-4.642454,0.954168,101.614319
3,ecmwf_ifs,2024-04-01 00:00:00,2024-04-01 01:00:00+00:00,1.7,2.06,4.98,129.0,110.0,1,2024-04-01 01:00:00+00:00,2304513,5.617938,105.831802,6,5.337706,0.157387,-5.108343,1.523224,106.603714
4,ecmwf_ifs,2024-04-01 00:00:00,2024-04-01 01:00:00+00:00,1.7,2.06,4.98,129.0,110.0,1,2024-04-01 01:00:00+00:00,2304514,4.995935,79.370956,6,4.735353,0.101319,-4.650571,-0.833310,79.841293


In [42]:
fdf[fdf['StationId'] == 2304518].shape

(61775, 19)